In [1]:
import rdflib
import sys; sys.path.append('..') 
import python_backend_server.constants as constants
from SPARQLWrapper import JSON, SPARQLWrapper
import pandas as pd

In [6]:
sparql = SPARQLWrapper(constants.VIRTUOSO_URL)
query_template = """
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?sensor ?time ?result
FROM <{graph}>
WHERE {{
    ?observation a sosa:Observation ;
                 sosa:resultTime ?time ;
                 sosa:madeBySensor ?sensor ;
                 sosa:hasSimpleResult ?result .
}}

"""
query = query_template.format(graph=constants.GRAPH_URI)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)

In [7]:
try:
    results = sparql.query().convert() #Execute the query and convert to JSON
    bindings = results["results"]["bindings"] #Parse the JSON into a list of dictionaries. Extracts the literal values and strips out RDF metadata types
    data = []
    for row in bindings:
        processed_row = {key: row[key]["value"] for key in row}
        data.append(processed_row)
    df = pd.DataFrame(data)
    print(df.head())

except Exception as e:
    print(f"An error occurred: {e}")

                                   sensor                       time  \
0  http://example.com/waterinfo/289429042  2021-08-13 05:00:00+00:00   
1  http://example.com/waterinfo/289429042  2021-08-13 05:15:00+00:00   
2  http://example.com/waterinfo/289429042  2021-08-13 05:30:00+00:00   
3  http://example.com/waterinfo/289429042  2021-08-13 05:45:00+00:00   
4  http://example.com/waterinfo/289429042  2021-08-13 06:00:00+00:00   

                  result  
0  5849.6599999999998545  
1                 5844.0  
2  5843.5200000000004366  
3  5850.0399999999999636  
4  5828.5799999999999272  


In [9]:
df = df.pivot(index='time', columns='sensor', values='result') #Reshape the DataFrame
df = df.reset_index() #Reset the index if you want 'time' back as a regular column

In [10]:
print(df.head())

sensor                       time http://example.com/waterinfo/111111111  \
0       2021-03-03 23:15:00+00:00                                    NaN   
1       2021-03-03 23:30:00+00:00                                    NaN   
2       2021-03-03 23:45:00+00:00                                    NaN   
3       2021-03-04 00:00:00+00:00                                    NaN   
4       2021-03-04 00:15:00+00:00                                    NaN   

sensor http://example.com/waterinfo/289423042  \
0                                         NaN   
1                                         NaN   
2                       902.26999999999998181   
3                                         NaN   
4                       901.96000000000003638   

sensor http://example.com/waterinfo/289429042  \
0                       1605.7699999999999818   
1                                         NaN   
2                       1602.3399999999999181   
3                       1611.2599999999999909   
4  

In [ ]:
df.to_csv("testttt.csv", index=False) #Save the first 5 rows to a CSV file